# Import Statements

In [1]:
import pandas as pd
df = pd.read_csv("2026dependenciesHWFile.csv")
# left_df = df[df.iloc[:6]]
left_df = df.iloc[:6091,:7]
right_df = df.iloc[:6091,8:].dropna()
right_df.rename(columns={'courseLetter.1': 'courseLetter', 'courseNumber.1': 'courseNumber'}, inplace=True)
right_df
# left_df

,courseDictionary,courseLetter,courseNumber
0,AME 105,AME,105
1,AME 121,AME,121
2,AME 122,AME,122
3,AME 205,AME,205
4,AME 264,AME,264
...,...,...,...
1360,WGS 310W,WGS,310W
1361,WGS 311W,WGS,311W
1362,WGS 330W,WGS,330W
1363,WGS 350W,WGS,350W


### Computing Courses

**Question 1**\
How many CIS courses at the 100 level are part of the Computer Information Systems, BS program?

In [2]:
left_df['courseNumber'] = left_df['courseNumber'].apply(str)

cis100_bs_df = left_df[
    (left_df['courseLetter'] == 'CIS') &
    (left_df['relatedTo'] == 'Computer Information Systems, BS') &
    (left_df['courseNumber'].str.startswith('1'))
    ]
cis100_bs_df.shape[0]

5

**Question 2**\
How many courses are offered with the CIS designation?

In [3]:
cis_total = (right_df['courseLetter'] == 'CIS').sum()
print(cis_total)

76


**Question 3**\
How many courses are offered with the CYB designation?

In [4]:
cyb_total = (right_df['courseLetter'] == 'CYB').sum()
print(cyb_total)

40


**Question 4**\
How many courses are offered with the DSC designation?

In [5]:
dsc_total = (right_df['courseLetter'] == 'DSC').sum()
print(dsc_total)

23


**Question 5**\
How many courses are offered with the ISS designation?

In [6]:
iss_total = (right_df['courseLetter'] == 'ISS').sum()
print(iss_total)

54


### Crosslisting of Courses

**Create a dataframe with course designated as "CrossList"**

In [7]:
crossList_df = left_df[left_df['How'] == 'CrossList']
crossList_df.shape[0]

376

**Question 6**\
Find how many cross lists each course has.  How may courses are crosslisted with POS 223?

In [8]:
cl_each_course_df = pd.DataFrame(
    [i, int((crossList_df['relatedTo'] == i).sum())] for i in left_df['relatedTo']
    )
cl_each_course_df.columns = ['course_name', 'cl']
pos_223_cl_total = (crossList_df['relatedTo'] == 'POS 223').sum()
print(pos_223_cl_total)
# print(cl_each_course_df['course_name'] == 'POS 223')
# cl_each_course_df

2


**Question 7**\
What is the mean number of crosslists per course?

In [9]:
mean_cl_per_course = cl_each_course_df['cl'].mean()
print(f"{mean_cl_per_course:.2f}")

0.22


**Question 8**\
What is the median number of courses crosslisted?

In [10]:
median_cl_per_course = cl_each_course_df['cl'].median()
print(f"{median_cl_per_course:.2f}")

0.00


**Question 9**\
What is the mode of the number of crosslisted courses?

In [11]:
print(cl_each_course_df['cl'].mode())

0    0
Name: cl, dtype: int64


**Question 10**\
Given that a course is actually crosslisted, what is the average number of courses it is crosslisted to?

In [12]:
course_cl = crossList_df.groupby('course')['relatedTo'].count()
cl_mean = course_cl.mean()
print(f"{cl_mean:.2f}")

1.39


**Question 11**\
How many courses have four designations (crosslisted designations count as 1 course in total)?  Note:  what would that look like in this dataset?

In [13]:
courses_with_four_designations = course_cl.value_counts()[3] / 4
print(int(courses_with_four_designations))

5


**Question 12**\
Assuming all of the crosslists are one course (e.g. CIS 255 and DSC 255 are one course), how many courses does UMA offer in its catalog?

In [14]:
total_cl_courses = course_cl.value_counts().sum()
print(total_cl_courses)
course_dict_total = right_df.value_counts().sum()
print(course_dict_total)
total_UMA_courses = course_dict_total - total_cl_courses
print(total_UMA_courses)

270
1365
1095


**Question 13**\
All four of the designations CIS, CYB, DSC, and ISS are offered by the computing group.  How many specific courses are offered by the computing group.  Note:  courses crosslisted together are ONE course for this purpose.

In [15]:
comp_group_df = left_df[left_df['courseLetter'].isin(['CIS', 'CYB', 'DSC', 'ISS'])]
comp_group_crosslist_df = comp_group_df[comp_group_df['How'] == 'CrossList']
# all_comp_cl_df.shape[0]
total_comp_cl = comp_group_crosslist_df.groupby('course')['relatedTo'].count()
print(total_comp_cl.value_counts())
print(total_comp_cl.value_counts().sum())
# comp_total = right_df[right_df['courseLetter'].isin(['CIS', 'CYB', 'DSC', 'ISS'])]
comp_total = right_df['courseLetter'].isin(['CIS', 'CYB', 'DSC', 'ISS']).sum()
print(comp_total)
# with pd.option_context('display.max_rows', None, 'display.max_columns', None):  # more options can be specified also
#     print(comp_group_df)

relatedTo
1    18
2    10
3     2
Name: count, dtype: int64
30
193


### Architecture Program
In this section, we will be studying the courses and programs offered by the Architecture faculty.

**Question 14**\
What proportion of courses (by code – crosslists are separate for this purpose) are in the Architecture, B.Arch checksheet?

In [16]:
arch_checksheet_total = (left_df['relatedTo'] == 'Architecture, B.Arch').sum()

prop_barch = (arch_checksheet_total / course_dict_total)

print(arch_checksheet_total)
print(course_dict_total)
print(f"{prop_barch:.2f}")

48
1365
0.04


**Question 15**\
What proportion of courses (by code) are ARC courses?

In [17]:
arc_total = (right_df['courseLetter'] == 'ARC').sum()
prop_arc = arc_total / course_dict_total
print(arc_total)
print(course_dict_total)
print(f"{prop_arc:.2f}")

38
1365
0.03


**Question 16**\
Assuming these are independent, what proportion of courses (by code) are both in the Architecture, B.Arch checksheet and are ARC courses?

In [20]:
barch_arc_df = df[
    (df['relatedTo'] == 'Architecture, B.Arch') &
    (df['courseLetter'] == 'ARC')
    ]

prop_barch_arc = len(barch_arc_df) / course_dict_total
print(prop_barch_arc)
print(f"{prop_barch_arc:.2f}")

0.021245421245421246
0.02


**Question 17**\
What is the conditional probability that a given course with an ARC designation is part of the Architecture, B.Arch checksheet?

In [24]:
# To answer this, P(A|B) or the probability of A given that B has happened. A = total courses with ARC designation, and B = Architecture, B.Arch.
# cp_barch_arc = len(arc_total_df) / len(barch_arc_df)
print(len(barch_arc_df))
print(arch_checksheet_total)
cp_barch_arc = len(barch_arc_df) / arch_checksheet_total
print(f"{cp_barch_arc:.2f}")

29
48
0.60


**Question 18**\
What is the conditional probability that a given course in the Architecture, B.Arch checksheet is an ARC course?

In [27]:
# To answer this, P(A|B) or the probability of A given that B has happened. A = total courses with ARC designation, and B = Architecture, B.Arch. checksheet.
# cp_arc_barch = len(arc_total_df) / len(arch_checksheet_df)
total_arc_courses = (left_df['courseLetter'] == 'ARC').sum()
print(len(barch_arc_df))
print(total_arc_courses)
cp_barch_arc = len(barch_arc_df) / total_arc_courses
print(f"{cp_barch_arc:.2f}")

29
115
0.25


**Question 19**\
Do the answers from 14-18 suggest that ARC courses and Architecture, B.Arch checksheet courses are independent statistically?  Why or why not.

### Prerequisites
In this section, we will address whether or not there is a bidirectional relationship over prerequisites (namely are courses equally likely to have prerequisites or to be prerequisites).

**Question 20**\
What proportion of courses (by code) are a prerequisite for another course?

In [28]:
prereq_total_df = df[
    (df['How'] == 'Prereq') &
    (df['relatedToCourseLetter'].notnull())
    ]

total_designations = df['course'].count()
print(len(prereq_total_df))
print(total_designations)
prereq_prop = len(prereq_total_df) / total_designations
print(round(prereq_prop, 2))

1722
6090
0.28


**Question 21**\
What proportion of courses (by code) have at least one prerequisite?

In [33]:
prereq_total = (left_df['How'] == 'Prereq').sum()
print(prereq_total)
print(course_dict_total)
print(total_designations)
prereq = prereq_total / total_designations
print(prereq)


1724
1365
6090
0.2830870279146141


In [30]:
prereq_total = df[df['How'] == 'Prereq']
total_designations = df['course'].count()
print(len(prereq_total))
print(total_designations)
prereq_prop = len(prereq_total) / total_designations
print(round(prereq_prop, 2))

1724
6090
0.28


In [29]:
course_id = (
    df.iloc[:, 1].astype(str).str.strip()
    + ' '
    + df.iloc[:, 2].astype(str).str.strip()
)

# 2. Denominator: Total unique course codes across the catalog
total_catalog_courses = course_id.nunique()

# 3. Numerator: Unique courses that have at least one prerequisite
type_col = 'How' if 'How' in df.columns else 'relationshipType'
prereq_mask = (
    df[type_col].astype(str).str.contains('pre', case=False, regex=False, na=False)
    & df['relatedTo'].notna()
    & ~df['relatedTo'].astype(str).str.strip().isin(['', 'nan', 'None'])
)

courses_with_prereqs = course_id[prereq_mask].nunique()

# 4. Proportion
proportion = courses_with_prereqs / total_catalog_courses

print(f"Courses with >= 1 prerequisite: {courses_with_prereqs}")
print(f"Total catalog unique courses:   {total_catalog_courses}")
print(f"Proportion:                     {proportion:.4f} ({proportion * 100:.2f}%)")

Courses with >= 1 prerequisite: 368
Total catalog unique courses:   1160
Proportion:                     0.3172 (31.72%)


### American Studies
The Curriculum Committeem has had a history of challenges with the American Studies program.  The coordinator of the program has what seems to be a liberal crosslisting policy.  This makes things difficult for those who maintain MaineStreet and maintain Acalog.  You will be addressing whether or not the data supports the claim of the Curriculum Committee that the coordinator of the program doth crosslist too much vis a vis their peers.

Recall:
An association rule has an antecedent and a consequent.  The antecedent begins the rule, while the consequent ends the rule.  If antecedent, then consequent.

**Support** is the proportion an item occurs in the dataset\
**Confidence** is the proportion of the time that given the antecedent, the consequent occurs.\
**Lift** is the ratio between the confidence of the rule and the support of its consequent.

**Question 22**\
What is the support for a course being an AME course?

In [ ]:
ame_total = (df['courseLetter'] == 'AME').sum()
print(ame_total)
ame_support = ame_total / total_designations
print(round(ame_support, 3))

**Question 23**\
What is the support for a course being crosslisted?

In [ ]:
total_cl = (df['How'] == 'CrossList').sum()
crossList_support = total_cl / total_designations
print(round(crossList_support, 3))

**Question 24**\
What is the confidence than an AME course is crosslisted?

In [ ]:
# Given that course is AME it is crosslisted
# To answer this, P(A|B) or the probability of A given that B has happened. A = crosslisted, and B =AME course
# ame_conf = total_cl / ame_total
ame_cl = df[
    (df['How'] == 'CrossList') &
    (df['courseLetter'] == 'AME')
    ]
ame_conf = len(ame_cl) / ame_total
print(round(ame_conf, 2))

**Question 25**\
What is the lift associated with knowing that a course is an AME course?

In [ ]:
# 2. Standardize prefix and unique course ID
# Using Column B (index 1) for primary courseLetter and Column C (index 2) for courseNumber
prefix = df.iloc[:, 1].astype(str).str.strip()
course_id = prefix + ' ' + df.iloc[:, 2].astype(str).str.strip()

# 3. Identify all crosslisted courses
# In this dataset, crosslist rows are marked in the 'How' or relationship column
type_col = 'How' if 'How' in df.columns else 'relationshipType'
cross_mask = df[type_col].astype(str).str.contains('cross', case=False, regex=False)

# Unique course codes that have at least one crosslist
crosslisted_courses = set(course_id[cross_mask].unique())

# 4. Define the universes
total_catalog_courses = course_id.nunique()
ame_mask = prefix == 'AME'
ame_courses = set(course_id[ame_mask].unique())

# 5. Calculate Support(B) = P(Crosslist)
support_b = len(crosslisted_courses) / total_catalog_courses

# 6. Calculate Confidence = P(Crosslist | AME)
ame_crosslisted = ame_courses.intersection(crosslisted_courses)
confidence = len(ame_crosslisted) / len(ame_courses) if len(ame_courses) > 0 else 0

# 7. Compute Lift
lift = confidence / support_b if support_b > 0 else 0

print(f"Total Unique Catalog Courses: {total_catalog_courses}")
print(f"Total Unique Crosslisted Courses: {len(crosslisted_courses)}")
print(f"Support(Crosslist) [P(B)]: {support_b:.4f}")
print(f"Total Unique AME Courses: {len(ame_courses)}")
print(f"AME Courses that are Crosslisted: {len(ame_crosslisted)}")
print(f"Confidence [P(Crosslist | AME)]: {confidence:.4f}")
print(f"Lift: {lift:.4f}")